# San Diego Neighborhood Opportunity Finder  
## Exploratory Data Analysis

I'm specifically looking into residential and mixed-use census tracts across San Diego County to identify neighborhoods that could have potential for residential development or community investment. It combines demographic, housing, safety, transportation, walkability, environmental risk, and school access data.

This notebook uses the final residential tract dataset created during the data wrangling and merge process.

Questions to answer:

- Which tracts look strong across several neighborhood factors?
- Which areas have better safety, walkability, transit, or school access?
- Which areas have higher climate or environmental risk?
- Are there tradeoffs between different neighborhood features?
- Which variables seem most useful for the final scoring system?

This notebook won’t create the final opportunity score yet. The goal is to understand the data, review feature relationships, and decide which variables should move forward into scoring.

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import geopandas as gpd

# prefered theme
sns.set_theme(style='whitegrid', palette='Set2')

# using this to show all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [2]:
# loading the final dataset used for EDA and scoring

file_path = '../data/processed/master_residential_tract_features.csv'

df = pd.read_csv(
    file_path,
    dtype={'tract_id': str}) # keep census tract ID as text so we don't accidentally treat it like a number or remove leading zeros

## Dataset Overview

In [3]:
df.shape

(727, 72)

In [4]:
df.head()

,safety_score,violent_safety_score,property_safety_score,walkability_index,jobs_housing_mix_score,employment_mix_score,intersection_density_score,commute_mode_diversity_score,transit_stop_density,has_transit_access,public_transit_commute_rate,no_vehicle_rate,climate_loss_risk_score,social_vulnerability_score,community_resilience_score,heat_risk_score,flood_risk_score,wildfire_risk_score,school_density,school_academic_score,tract_id,tract_name,total_population,tract_type_flag,population_under_5_rate,population_under_18_rate,median_age,hispanic_latino_rate,total_households,households_with_children,avg_household_size,avg_family_size,bachelors_or_higher,median_household_income,poverty_rate,family_poverty_rate,unemployment_rate,drove_alone_rate,work_from_home_rate,renter_rate,median_gross_rent,rent_burden_30_34_count,rent_burden_35_plus_count,vacancy_rate,total_crime_count,violent_crime_count,property_crime_count,crime_rate_per_1000,violent_crime_rate_per_1000,property_crime_rate_per_1000,low_population_flag,transit_stop_count,large_tract_flag,expected_annual_loss_rating_composite,social_vulnerability_rating,community_resilience_rating,heat_wave_hazard_type_risk_index_rating,inland_flooding_hazard_type_risk_index_rating,wildfire_hazard_type_risk_index_rating,school_count,elementary_school_count,middle_school_count,high_school_count,charter_school_count,district_name,district_type,district_grade_low,district_grade_high,district_overlap_pct,academic_strength_tier,missing_walkability_flag,exclude_from_scoring_flag
0,31.241473,37.926330,18.826739,14.740561,5.226528,10.763785,17.556632,18.669896,10.115410,1,1.8,3.5,58.092397,27.460167,5.377325,16.454550,41.347080,38.169646,0.000000,3.5,06073000100,Census Tract 1; San Diego County; California,2948,residential_or_mixed,5.9,22.3,51.1,9.5,1178,318,2.50,2.89,1758,231667.0,2.2,2.1,0.0,69.8,18.9,9.4,NaN,17,20,8.9,74,2,38,25.101764,0.678426,12.890095,False,6,0,Relatively Moderate,Relatively Low,Very Low,Very Low,Relatively Low,Very Low,0,0,0,0,0,San Diego Unified,Unified,PK,12,1.0,medium_high,0,0
1,27.066849,30.695771,20.327422,18.000000,16.000000,20.000000,17.000000,19.000000,20.978476,1,0.7,12.5,21.362473,46.128650,5.377325,12.578054,9.441927,39.606150,0.000000,3.5,06073000201,Census Tract 2.01; San Diego County; California,2270,residential_or_mixed,3.5,19.4,51.2,4.0,1180,208,1.92,3.03,1392,124722.0,6.7,6.5,2.3,51.0,34.3,54.7,2407.0,59,233,6.4,60,3,26,26.431718,1.321586,11.453744,False,7,0,Relatively Low,Relatively Moderate,Very Low,Very Low,Very Low,Very Low,0,0,0,0,0,San Diego Unified,Unified,PK,12,1.0,medium_high,0,0
2,13.335607,13.096862,13.778990,15.573006,9.379487,14.026923,17.648718,17.367094,21.814380,1,4.8,4.4,50.467388,56.890955,5.377325,33.510892,41.477888,0.000000,1.983143,3.5,06073000202,Census Tract 2.02; San Diego County; California,3755,residential_or_mixed,2.8,7.9,45.8,15.0,2240,185,1.64,2.44,2077,120091.0,2.9,0.9,5.0,58.0,33.2,55.2,1992.0,134,375,9.5,186,16,64,49.533955,4.260985,17.043941,False,11,0,Relatively Low,Relatively Moderate,Very Low,Relatively Low,Relatively Low,No Rating,1,1,0,0,0,San Diego Unified,Unified,PK,12,1.0,medium_high,0,0
3,21.002729,26.875853,10.095498,NaN,NaN,NaN,NaN,NaN,45.122430,1,5.9,6.2,15.711534,9.209856,5.377325,9.806235,5.858989,0.000000,0.000000,3.5,06073000301,Census Tract 3.01; San Diego County; California,2311,residential_or_mixed,4.0,8.1,37.3,22.3,1302,131,1.77,2.55,1248,87813.0,15.4,18.1,4.1,56.9,31.0,73.5,1945.0,69,316,11.7,151,4,47,65.339680,1.730852,20.337516,False,7,0,Very Low,Very Low,Very Low,Very Low,Very Low,No Rating,0,0,0,0,0,San Diego Unified,Unified,PK,12,1.0,medium_high,1,0
4,7.175989,8.321965,5.047749,NaN,NaN,NaN,NaN,NaN,52.115092,1,4.2,6.7,40.969303,58.817153,5.377325,21.212880,16.762394,0.000000,0.000000,3.5,06073000302,Census Tract 3.02; San Diego County; California,2873,residential_or_mixed,0.0,2.5,45.1,15.7,1818,63,1.48,1.99,1644,89573.0,9.4,0.0,2.3,49.1,27.8,72.4,2412.0,80,664,17.3,271,18,99,94.326488,6.265228,34.458754

In [5]:
df.columns.tolist()

['safety_score',
 'violent_safety_score',
 'property_safety_score',
 'walkability_index',
 'jobs_housing_mix_score',
 'employment_mix_score',
 'intersection_density_score',
 'commute_mode_diversity_score',
 'transit_stop_density',
 'has_transit_access',
 'public_transit_commute_rate',
 'no_vehicle_rate',
 'climate_loss_risk_score',
 'social_vulnerability_score',
 'community_resilience_score',
 'heat_risk_score',
 'flood_risk_score',
 'wildfire_risk_score',
 'school_density',
 'school_academic_score',
 'tract_id',
 'tract_name',
 'total_population',
 'tract_type_flag',
 'population_under_5_rate',
 'population_under_18_rate',
 'median_age',
 'hispanic_latino_rate',
 'total_households',
 'households_with_children',
 'avg_household_size',
 'avg_family_size',
 'bachelors_or_higher',
 'median_household_income',
 'poverty_rate',
 'family_poverty_rate',
 'unemployment_rate',
 'drove_alone_rate',
 'work_from_home_rate',
 'renter_rate',
 'median_gross_rent',
 'rent_burden_30_34_count',
 'rent_bu

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 727 entries, 0 to 726
Data columns (total 72 columns):
 #   Column                                         Non-Null Count  Dtype  
---  ------                                         --------------  -----  
 0   safety_score                                   727 non-null    float64
 1   violent_safety_score                           727 non-null    float64
 2   property_safety_score                          727 non-null    float64
 3   walkability_index                              528 non-null    float64
 4   jobs_housing_mix_score                         528 non-null    float64
 5   employment_mix_score                           528 non-null    float64
 6   intersection_density_score                     528 non-null    float64
 7   commute_mode_diversity_score                   528 non-null    float64
 8   transit_stop_density                           727 non-null    float64
 9   has_transit_access                             727 non-null    in

## Checking Null Values in Walkability

Before exploring patterns, I’ll review the remaining missing values and confirm that each tract appears only once.

Most of the cleaning was completed in the earlier notebooks, so this section is mainly checking for issues that could affect the EDA or final scoring.

In [12]:
# setting the walkability data path

raw_dir = Path('../data/raw/WalkabilityIndex')
walkability_path = raw_dir / 'Natl_WI.gdb'

In [11]:
# checking the available layers

gpd.list_layers(walkability_path)

,name,geometry_type
0,NationalWalkabilityIndex,MultiPolygon


In [16]:
# loading the walkability geography

walk_gdf = gpd.read_file(
    walkability_path,
    layer='NationalWalkabilityIndex')

walk_gdf.shape

(220739, 30)

In [17]:
walk_gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 220739 entries, 0 to 220738
Data columns (total 30 columns):
 #   Column        Non-Null Count   Dtype   
---  ------        --------------   -----   
 0   GEOID10       220739 non-null  str     
 1   GEOID20       220739 non-null  str     
 2   STATEFP       220739 non-null  str     
 3   COUNTYFP      220739 non-null  str     
 4   TRACTCE       220739 non-null  str     
 5   BLKGRPCE      220739 non-null  str     
 6   CSA           167709 non-null  str     
 7   CSA_Name      167709 non-null  str     
 8   CBSA          220739 non-null  str     
 9   CBSA_Name     203645 non-null  str     
 10  Ac_Total      220739 non-null  float64 
 11  Ac_Water      220739 non-null  float64 
 12  Ac_Land       220739 non-null  float64 
 13  Ac_Unpr       220739 non-null  float64 
 14  TotPop        220739 non-null  int32   
 15  CountHU       220464 non-null  float64 
 16  HH            220464 non-null  float64 
 17  Workers       220739 

In [18]:
# filtering to San Diego County

sd_walk_gdf = walk_gdf[
    walk_gdf['GEOID20'].str.startswith('06073')].copy()

sd_walk_gdf['tract_id'] = sd_walk_gdf['GEOID20'].str[:11]

sd_walk_gdf.shape

(1795, 31)

In [19]:
# combining block groups into tract boundaries

tract_gdf = (
    sd_walk_gdf[['tract_id', 'geometry']]
    .dissolve(by='tract_id')
    .reset_index())

tract_gdf.shape

(628, 2)

In [21]:
from pathlib import Path
import geopandas as gpd

tract_path = Path('../data/raw/census_tracts/tl_2024_06_tract.shp')

tract_gdf = gpd.read_file(tract_path)

tract_gdf.shape

(9129, 14)

In [22]:
# filtering to San Diego County tracts

sd_tract_gdf = tract_gdf[
    tract_gdf['COUNTYFP'] == '073'].copy()

sd_tract_gdf.shape

(737, 14)

In [23]:
sd_tract_gdf[['GEOID', 'NAME']].head()

,GEOID,NAME
817,06073008331,83.31
818,06073008336,83.36
819,06073008337,83.37
820,06073011601,116.01
821,06073011602,116.02


In [24]:
print(sd_tract_gdf.crs)
print(sd_walk_gdf.crs)

EPSG:4269
PROJCS["USA_Contiguous_Albers_Equal_Area_Conic_USGS_version",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["ESRI","102039"]]


In [25]:
# matching the tract boundaries to the walkability CRS

sd_tract_gdf = sd_tract_gdf.to_crs(sd_walk_gdf.crs)

print(sd_tract_gdf.crs)
print(sd_walk_gdf.crs)

PROJCS["USA_Contiguous_Albers_Equal_Area_Conic_USGS_version",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["ESRI","102039"]]
PROJCS["USA_Contiguous_Albers_Equal_Area_Conic_USGS_version",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122

In [27]:
# renaming the tract ID before the spatial overlay

sd_tract_gdf = sd_tract_gdf.rename(
    columns={'GEOID': 'tract_id'})

sd_tract_gdf[['tract_id', 'NAME', 'geometry']].head()

,tract_id,NAME,geometry
817,06073008331,83.31,"POLYGON ((-1956765.703 1316282.945, -1956756.3..."
818,06073008336,83.36,"POLYGON ((-1947669.256 1317369.017, -1947665.7..."
819,06073008337,83.37,"POLYGON ((-1948823.66 1315993.566, -1948798.24..."
820,06073011601,116.01,"POLYGON ((-1951977.102 1283790.452, -1951936.6..."
821,06073011602,116.02,"POLYGON ((-1951920.828 1283185.98, -1951900.43..."


In [28]:
# keeping the EPA walkability fields used in the final dataset

walk_cols = [
    'GEOID20',
    'NatWalkInd',
    'D2A_Ranked',
    'D2B_Ranked',
    'D3B_Ranked',
    'D4A_Ranked',
    'geometry']

walk_overlay_gdf = sd_walk_gdf[walk_cols].copy()

In [29]:
# overlaying the EPA block groups with the 2024 tract boundaries

walk_tract_overlap = gpd.overlay(
    walk_overlay_gdf,
    sd_tract_gdf[['tract_id', 'geometry']],
    how='intersection')

walk_tract_overlap.shape

(7576, 8)

In [30]:
# calculating how much of each EPA polygon falls inside each tract

walk_tract_overlap['overlap_area'] = (
    walk_tract_overlap.geometry.area)

walk_tract_overlap[
    ['tract_id', 'GEOID20', 'overlap_area']].head()

,tract_id,GEOID20,overlap_area
0,06073017069,060730170341,1.288818e-03
1,06073017071,060730170341,1.634613e-02
2,06073017033,060730170341,1.139318e-02
3,06073017018,060730170341,1.692401e-03
4,06073017034,060730170341,1.918283e+06


In [31]:
# keeping the walkability fields and household weights

walk_cols = [
    'GEOID20',
    'HH',
    'NatWalkInd',
    'D2A_Ranked',
    'D2B_Ranked',
    'D3B_Ranked',
    'D4A_Ranked',
    'geometry']

walk_overlay_gdf = sd_walk_gdf[walk_cols].copy()

In [32]:
# saving the original EPA polygon area before the overlay

walk_overlay_gdf['source_area'] = walk_overlay_gdf.geometry.area

walk_tract_overlap = gpd.overlay(
    walk_overlay_gdf,
    sd_tract_gdf[['tract_id', 'geometry']],
    how='intersection')

walk_tract_overlap['overlap_area'] = walk_tract_overlap.geometry.area
walk_tract_overlap['overlap_pct'] = (
    walk_tract_overlap['overlap_area']
    / walk_tract_overlap['source_area'])

In [33]:
walk_tract_overlap['overlap_pct'].describe()

count    7.576000e+03
mean     2.369317e-01
std      4.220688e-01
min      9.081361e-20
25%      5.538093e-09
50%      1.786077e-08
75%      1.814644e-02
max      1.000000e+00
Name: overlap_pct, dtype: float64

Because the boundaries in the initial merge didn't line up exactly, I had to do a spatial overlay. It shows which older walkability areas fall inside each newer tract, then rebuild a walkability score for the 2024 tract. This was more accurate than imputing an average from nearby tracts.

<b>Process</b>

I selected the EPA walkability fields including household counts, then matched the EPA block-group polygons to the updated 2024 census tract boundaries. The overlay split polygons wherever the two boundary systems crossed. After that, I calculated how much of each EPA block group falls inside each 2024 tract. The summary shows a lot of slim overlaps, so we’ll need to remove those small slivers before calculating the final weighted walkability values.

In [34]:
# removing tiny overlaps that only touch tract edges

walk_tract_overlap = walk_tract_overlap[
    walk_tract_overlap['overlap_pct'] >= 0.01].copy()

walk_tract_overlap.shape

(1926, 12)

In [36]:
# estimating how many households fall inside each tract overlap

walk_tract_overlap['adjusted_hh'] = (
    walk_tract_overlap['HH']
    * walk_tract_overlap['overlap_pct'])

walk_tract_overlap[
    ['tract_id', 'GEOID20', 'HH', 'overlap_pct', 'adjusted_hh']].sort_values(by='overlap_pct', ascending=False).head()

,tract_id,GEOID20,HH,overlap_pct,adjusted_hh
6018,06073010009,060730100092,466.0,1.0,466.0
7399,06073020304,060730203042,208.0,1.0,208.0
587,06073008501,060730085011,685.0,1.0,685.0
6257,06073016901,060730169013,427.0,1.0,427.0
5699,06073008362,060730083623,600.0,1.0,600.0


In [37]:
# calculating household-weighted walkability values by tract

walk_features = [
    'NatWalkInd',
    'D2A_Ranked',
    'D2B_Ranked',
    'D3B_Ranked',
    'D4A_Ranked']

for col in walk_features:
    walk_tract_overlap[f'{col}_weighted'] = (
        walk_tract_overlap[col]
        * walk_tract_overlap['adjusted_hh'])

In [38]:
# combining the overlap pieces into one row per tract

walk_2024 = (
    walk_tract_overlap
    .groupby('tract_id')
    .agg(
        total_adjusted_hh=('adjusted_hh', 'sum'),
        NatWalkInd_weighted=('NatWalkInd_weighted', 'sum'),
        D2A_Ranked_weighted=('D2A_Ranked_weighted', 'sum'),
        D2B_Ranked_weighted=('D2B_Ranked_weighted', 'sum'),
        D3B_Ranked_weighted=('D3B_Ranked_weighted', 'sum'),
        D4A_Ranked_weighted=('D4A_Ranked_weighted', 'sum'))
    .reset_index())

walk_2024.shape

(737, 7)

This grouped all of the smaller overlap pieces back into one row for each 2024 census tract. For every tract, it added up the estimated households and the weighted walkability values. Now there are 737 rows, which matches the number of San Diego County tracts. I still need to divide each weighted total by the total adjusted households to get the final average walkability scores.

In [39]:
# calculating the final average walkability values for each tract

for col in walk_features:
    walk_2024[col] = (
        walk_2024[f'{col}_weighted']
        / walk_2024['total_adjusted_hh'])

walk_2024[
    ['tract_id', 'NatWalkInd', 'D2A_Ranked', 'D2B_Ranked', 'D3B_Ranked', 'D4A_Ranked']].head()

,tract_id,NatWalkInd,D2A_Ranked,D2B_Ranked,D3B_Ranked,D4A_Ranked
0,06073000100,14.740238,5.234290,10.738557,17.558573,18.675718
1,06073000201,18.000000,16.000000,20.000000,17.000000,19.000000
2,06073000202,15.573006,9.379487,14.026923,17.648718,17.367094
3,06073000301,14.547884,8.397322,9.512863,16.000000,18.688559
4,06073000302,16.098350,14.517801,9.930760,17.442156,18.628614


In [40]:
# renaming the recalculated walkability fields

walk_2024 = walk_2024.rename(
    columns={
        'NatWalkInd': 'walkability_index_new',
        'D2A_Ranked': 'jobs_housing_mix_score_new',
        'D2B_Ranked': 'employment_mix_score_new',
        'D3B_Ranked': 'intersection_density_score_new',
        'D4A_Ranked': 'commute_mode_diversity_score_new'})

walk_2024[
    [
        'tract_id',
        'walkability_index_new',
        'jobs_housing_mix_score_new',
        'employment_mix_score_new',
        'intersection_density_score_new',
        'commute_mode_diversity_score_new']].head()

,tract_id,walkability_index_new,jobs_housing_mix_score_new,employment_mix_score_new,intersection_density_score_new,commute_mode_diversity_score_new
0,06073000100,14.740238,5.234290,10.738557,17.558573,18.675718
1,06073000201,18.000000,16.000000,20.000000,17.000000,19.000000
2,06073000202,15.573006,9.379487,14.026923,17.648718,17.367094
3,06073000301,14.547884,8.397322,9.512863,16.000000,18.688559
4,06073000302,16.098350,14.517801,9.930760,17.442156,18.628614


In [41]:
# merging the recalculated walkability values into the main dataset

walk_new_cols = [
    'tract_id',
    'walkability_index_new',
    'jobs_housing_mix_score_new',
    'employment_mix_score_new',
    'intersection_density_score_new',
    'commute_mode_diversity_score_new']

df = df.merge(
    walk_2024[walk_new_cols],
    on='tract_id',
    how='left')

df[
    [
        'tract_id',
        'walkability_index',
        'walkability_index_new',
        'missing_walkability_flag']].head()

,tract_id,walkability_index,walkability_index_new,missing_walkability_flag
0,06073000100,14.740561,14.740238,0
1,06073000201,18.000000,18.000000,0
2,06073000202,15.573006,15.573006,0
3,06073000301,NaN,14.547884,1
4,06073000302,NaN,16.098350,1


In [42]:
# filling only the missing walkability values

walk_pairs = {
    'walkability_index': 'walkability_index_new',
    'jobs_housing_mix_score': 'jobs_housing_mix_score_new',
    'employment_mix_score': 'employment_mix_score_new',
    'intersection_density_score': 'intersection_density_score_new',
    'commute_mode_diversity_score': 'commute_mode_diversity_score_new'}

for old_col, new_col in walk_pairs.items():
    df[old_col] = df[old_col].fillna(df[new_col])

df[list(walk_pairs.keys())].isna().sum()

walkability_index               0
jobs_housing_mix_score          0
employment_mix_score            0
intersection_density_score      0
commute_mode_diversity_score    0
dtype: int64

In [43]:
# marking tracts where walkability values were recalculated

df['walkability_imputed_flag'] = df['missing_walkability_flag']

df['walkability_imputed_flag'].value_counts()

walkability_imputed_flag
0    528
1    199
Name: count, dtype: int64

## Walkability Null Values

In retrospect, I should have done this step in the data wrangling phase when I saw that there were a lot of null values.

The missing walkability values came from a mismatch between the older EPA block-group boundaries and the newer 2024 census tract boundaries. I used a spatial overlay to match the two boundary types and recalculate walkability values for each 2024 tract.

In this process, I filled 199 missing tract values. I also kept a flag showing which values were recalculated so I can track them later during scoring and interpretation if necessary.

## Remaining Null Values

After recalculating the missing walkability values, I'm addressing the remaining null avlues. 

I’m not filling anything else yet. I want to review the remaining gaps first and decide whether they should be imputed, left missing, or kept as context only.

In [44]:
# checking the remaining null values

null_summary = (
    df.isna()
    .sum()
    .to_frame('missing_count'))

null_summary['missing_pct'] = (
    null_summary['missing_count'] / len(df) * 100).round(2)

null_summary = (
    null_summary[
        null_summary['missing_count'] > 0]
    .sort_values('missing_pct', ascending=False))

null_summary

,missing_count,missing_pct
median_gross_rent,79,10.87
median_household_income,8,1.10


## Missing Median Gross Rent

`median_gross_rent` is missing for 79 tracts, or about 10.9% of the residential and mixed-use dataset.

Before estimating those values, I want to check whether the missing rent is connected to low renter rates, income, poverty, vacancy, or other factors.

I think I'll leave the household income null since it's close to 1% null values, but I'll address median rent because I might want to address rental properties for development. 

In [46]:
# reviewing tracts with missing rent

missing_rent = df[
    df['median_gross_rent'].isna()].copy()

missing_rent[
    [
        'tract_id',
        'tract_name',
        'total_population',
        'total_households',
        'renter_rate',
        'median_household_income',
        'poverty_rate',
        'vacancy_rate',
        'tract_type_flag']].head(20)

,tract_id,tract_name,total_population,total_households,renter_rate,median_household_income,poverty_rate,vacancy_rate,tract_type_flag
0,06073000100,Census Tract 1; San Diego County; California,2948,1178,9.4,231667.0,2.2,8.9,residential_or_mixed
123,06073006600,Census Tract 66; San Diego County; California,2032,477,100.0,110651.0,6.0,26.6,residential_or_mixed
127,06073007002,Census Tract 70.02; San Diego County; California,3025,1256,11.8,184167.0,4.6,3.2,residential_or_mixed
130,06073007302,Census Tract 73.02; San Diego County; California,2610,984,20.7,183529.0,2.1,4.6,residential_or_mixed
154,06073008202,Census Tract 82.02; San Diego County; California,1203,659,63.7,111836.0,3.9,38.3,residential_or_mixed
155,06073008301,Census Tract 83.01; San Diego County; California,3190,1347,16.9,216823.0,2.4,0.0,residential_or_mixed
156,06073008303,Census Tract 83.03; San Diego County; California,3055,1341,20.1,230511.0,6.1,19.7,residential_or_mixed
158,06073008306,Census Tract 83.06; San Diego County; California,2711,1092,8.0,204779.0,9.1,2.1,residential_or_mixed
160,06073008310,Census Tract 83.10; San Diego County; California,6538,2457,7.5,172399.0,2.4,4.2,residential_or_mixed
161,06073008311,Census Tract 83.11; San Diego County; California,2739,1088,4.7,NaN,3.3,9.0,residential_or_mixed


In [47]:
# summarizing the missing-rent tracts

missing_rent[
    [
        'total_population',
        'total_households',
        'renter_rate',
        'median_household_income',
        'poverty_rate',
        'vacancy_rate']].describe().round(2)

,total_population,total_households,renter_rate,median_household_income,poverty_rate,vacancy_rate
count,79.00,79.00,79.00,71.00,79.00,79.00
mean,3925.77,1373.42,22.56,175152.49,4.85,8.04
std,1569.46,556.62,23.22,44067.22,3.74,8.59
min,1203.00,477.00,2.30,60455.00,0.60,0.00
25%,2725.00,998.50,7.85,152982.00,2.35,2.05
50%,3663.00,1256.00,14.10,183529.00,4.20,5.60
75%,4689.50,1593.00,25.90,208977.00,6.05,9.70
max,8172.00,2938.00,100.00,247222.00,28.20,38.30


It looks like rent is missing in higher-income, primarily owner-occupied tracts with less rental households. The missing values are probably not random, so I have to figure out how to impute. 

In [48]:
# comparing tracts with and without observed rent

df['rent_missing_flag'] = df['median_gross_rent'].isna().astype(int)

rent_missing_compare = (
    df.groupby('rent_missing_flag')[
        [
            'renter_rate',
            'median_household_income',
            'poverty_rate',
            'vacancy_rate',
            'total_population',
            'total_households']].median().round(2))

rent_missing_compare

,renter_rate,median_household_income,poverty_rate,vacancy_rate,total_population,total_households
rent_missing_flag,,,,,,
0,45.2,104849.5,8.7,4.6,4344.5,1582.0
1,14.1,183529.0,4.2,5.6,3663.0,1256.0


In [49]:
# checking which features are most related to observed rent

rent_relationships = (
    df[
        [
            'median_gross_rent',
            'median_household_income',
            'renter_rate',
            'poverty_rate',
            'vacancy_rate'
        ]
    ]
    .corr()['median_gross_rent']
    .drop('median_gross_rent')
    .sort_values(ascending=False))

rent_relationships

median_household_income    0.632264
vacancy_rate              -0.029759
renter_rate               -0.310573
poverty_rate              -0.431376
Name: median_gross_rent, dtype: float64

Income has the strongest relationship with observed rent (0.63). Poverty also has a moderate negative relationship (-0.43). Renter rate and vacancy rate are weaker.

In [50]:
# creating income groups from tracts with observed income

df['income_group'] = pd.qcut(
    df['median_household_income'],
    q=5,
    duplicates='drop')

df.groupby('income_group', observed=False)[
    'median_gross_rent'].median().round(2)

income_group
(34147.999, 76547.4]    1839.5
(76547.4, 98557.2]      2073.0
(98557.2, 118568.0]     2340.0
(118568.0, 146139.0]    2606.0
(146139.0, 247222.0]    2941.5
Name: median_gross_rent, dtype: float64

In [51]:
# creating a separate estimated rent column

income_group_rent = (
    df.groupby('income_group', observed=False)['median_gross_rent']
    .transform('median'))

df['estimated_median_gross_rent'] = (
    df['median_gross_rent']
    .fillna(income_group_rent))

df[
    [
        'median_gross_rent',
        'estimated_median_gross_rent'
    ]].isna().sum()

median_gross_rent              79
estimated_median_gross_rent     8
dtype: int64